# Day 4: Neural Networks + PyTorch

Exercises:
1. 2-layer neural network from scratch (NumPy) — backprop on XOR
2. Activation function comparison — vanishing gradients
3. Reimplement in PyTorch
4. MNIST with PyTorch
5. Overfit one batch sanity check

In [4]:
import numpy as np
import matplotlib.pyplot as plt

---
## Exercise 1: 2-Layer Neural Network from Scratch

Implement a 2-layer network in NumPy and train it on XOR.

In [5]:
# XOR data
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

In [18]:
# Implement a 2-layer network:
#   Layer 1: X (4,2) @ W1 (2, hidden) + b1 → ReLU → H (4, hidden)
#   Layer 2: H (4, hidden) @ W2 (hidden, 1) + b2 → sigmoid → output (4, 1)
#   Loss: BCE
#
# Forward pass:
#   z1 = X @ W1 + b1
#   h = relu(z1)          (or sigmoid — you'll swap in exercise 2)
#   z2 = h @ W2 + b2
#   y_hat = sigmoid(z2)
#
# Backward pass (derive these from chain rule):
#   dL/dz2 = y_hat - y
#   dL/dW2 = h.T @ dL/dz2
#   dL/db2 = sum(dL/dz2)
#   dL/dh = dL/dz2 @ W2.T
#   dL/dz1 = dL/dh * relu_derivative(z1)
#   dL/dW1 = X.T @ dL/dz1
#   dL/db1 = sum(dL/dz1)
#
# Use hidden_size=8, lr=0.1, train for 10000 iterations
# Print final predictions — they should be close to [0, 1, 1, 0]

# Your code here:

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def forward_pass(X, W1, W2, b1, b2):
    z1 = X @ W1 + b1
    h = relu(z1)
    z2 = h @ W2 + b2
    return z1, h, sigmoid(z2)

def backward_pass(X, y, W1, W2, b1, b2, iterations=1000, alpha=.01):

    for i in range(iterations):
        #   dL/dz2 = y_hat - y
        #   dL/dW2 = h.T @ dL/dz2
        #   dL/db2 = sum(dL/dz2)
        #   dL/dh = dL/dz2 @ W2.T
        #   dL/dz1 = dL/dh * relu_derivative(z1)
        #   dL/dW1 = X.T @ dL/dz1
        #   dL/db1 = sum(dL/dz1)
    
        # make prediction
        z1, h, y_hat = forward_pass(X, W1, W2, b1, b2)
    
        # compute the gradients
        dL_dz2 = y_hat - y
        dL_dW2 = h.T @ dL_dz2
        dL_db2 = np.sum(dL_dz2)
        dL_dh  = dL_dz2 @ W2.T
        dL_dz1 = dL_dh * relu_derivative(z1)
        dL_dW1 = X.T @ dL_dz1
        dL_db1 = np.sum(dL_dz1)

        # update the parameters
        W1 -= (alpha * dL_dW1)
        b1 -= (alpha * dL_db1)
        W2 -= (alpha * dL_dW2)
        b2 -= (alpha * dL_db2)

    return W1, W2, b1, b2

def print_parameters(W1, W2, b1, b2):
    print(f"W1: {W1}")
    print(f"b1: {b1}")
    print(f"W2: {W2}")
    print(f"b2: {b2}")

W1 = np.random.randn(2, 8) * 0.5
b1 = np.zeros((1, 8))
W2 = np.random.randn(8, 1) * 0.5
b2 = np.zeros((1, 1))

W1, W2, b1, b2 = backward_pass(X_xor, y_xor, W1, W2, b1, b2)
print_parameters(W1, W2, b1, b2)


# show that it works
print("=" * 50 + "\n")
print("Testing the XOR Neural Network")
_, _, results = forward_pass(X_xor, W1, W2, b1, b2)
print(f"Results: {results}")

W1: [[ 0.966841   -1.62698015  0.95984636 -0.34679342  0.84002756 -0.92600468
   1.75915633 -0.02108901]
 [ 0.13692703  1.62754119 -0.95834294 -0.5061065   0.07998107  0.92631076
  -1.75962839 -0.07853402]]
b1: [[0.00215489 0.00215489 0.00215489 0.00215489 0.00215489 0.00215489
  0.00215489 0.00215489]]
W2: [[-0.64411914]
 [ 2.2151401 ]
 [ 1.51185486]
 [ 0.01189099]
 [-0.27890622]
 [ 1.13616484]
 [ 2.29034751]
 [-0.07868351]]
b2: [[-1.62919604]]

Testing the XOR Neural Network
Results: [[0.16576915]
 [0.94898172]
 [0.95258254]
 [0.07040125]]


## 
Exercise 2: Activation Function Comparison

Swap sigmoid for ReLU in the hidden layer and compare training.

In [23]:
# Train the same XOR network twice:
#   1. Hidden activation = sigmoid
#   2. Hidden activation = ReLU
#
# For each, record the loss at every iteration.
# Plot both loss curves on the same chart.
# Observe: sigmoid trains slower (vanishing gradients flatten the loss curve)

# Your code here:

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid_derivative(x):
    return sigmoid(x) * (1 - sigmoid(x))

def forward_pass(X, W1, W2, b1, b2, sigmoid_act=False):
    z1 = X @ W1 + b1
    h = sigmoid(z1) if sigmoid_act else relu(z1)
    z2 = h @ W2 + b2
    return z1, h, sigmoid(z2)

def backward_pass(X, y, W1, W2, b1, b2, sigmoid_act=False, iterations=1000, alpha=.01):

    for i in range(iterations):
        #   dL/dz2 = y_hat - y
        #   dL/dW2 = h.T @ dL/dz2
        #   dL/db2 = sum(dL/dz2)
        #   dL/dh = dL/dz2 @ W2.T
        #   dL/dz1 = dL/dh * relu_derivative(z1)
        #   dL/dW1 = X.T @ dL/dz1
        #   dL/db1 = sum(dL/dz1)
    
        # make prediction
        z1, h, y_hat = forward_pass(X, W1, W2, b1, b2, sigmoid_act=sigmoid_act)
    
        # compute the gradients
        dL_dz2 = y_hat - y
        dL_dW2 = h.T @ dL_dz2
        dL_db2 = np.sum(dL_dz2)
        dL_dh  = dL_dz2 @ W2.T
        
        activation_fnc_derivative = sigmoid_derivative(z1) if sigmoid_act else relu_derivative(z1)
        
        dL_dz1 = dL_dh * activation_fnc_derivative
        dL_dW1 = X.T @ dL_dz1
        dL_db1 = np.sum(dL_dz1)

        # update the parameters
        W1 -= (alpha * dL_dW1)
        b1 -= (alpha * dL_db1)
        W2 -= (alpha * dL_dW2)
        b2 -= (alpha * dL_db2)

    return W1, W2, b1, b2


W1, W2, b1, b2 = backward_pass(X_xor, y_xor, W1, W2, b1, b2, sigmoid_act=True)
#print_parameters(W1, W2, b1, b2)

# Compare Results
print("=" * 50 + "\n")
print("XOR Neural Network: (Hidden Activation = Sigmoid)")
_, _, results = forward_pass(X_xor, W1, W2, b1, b2, sigmoid_act=True)
print(f"Sigmoid Results: {results}")


W1, W2, b1, b2 = backward_pass(X_xor, y_xor, W1, W2, b1, b2)
#print_parameters(W1, W2, b1, b2)
print("=" * 50 + "\n")
print("XOR Neural Network: (Hidden Activation = ReLU)")
_, _, results = forward_pass(X_xor, W1, W2, b1, b2)
print(f"ReLU Results: {results}")



XOR Neural Network: (Hidden Activation = Sigmoid)
Sigmoid Results: [[0.19517805]
 [0.83956024]
 [0.84921999]
 [0.15264487]]

XOR Neural Network: (Hidden Activation = ReLU)
ReLU Results: [[0.02944998]
 [0.99927343]
 [0.99941613]
 [0.02148882]]


---
## Exercise 3: Reimplement in PyTorch

Build the same XOR network using `nn.Module`.

In [41]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define a class that extends nn.Module:
#   __init__: define layers (nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
#   forward: pass input through the layers
#
# Training loop:
#   loss_fn = nn.BCELoss()
#   optimizer = optim.SGD(model.parameters(), lr=0.1)
#   for epoch in range(10000):
#       y_hat = model(X)
#       loss = loss_fn(y_hat, y)
#       optimizer.zero_grad()
#       loss.backward()
#       optimizer.step()
#
# Compare: how much shorter is this vs your NumPy version?

# Your code here:

class XOR_Neural_Network(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        # initialize the layers
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layer = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        z1 = self.input_layer(x)
        h = torch.relu(z1)
        z2 = self.hidden_layer(h)
        return self.sigmoid(z2)
    

# Model setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found!")
else:
    device = torch.device("cpu")
    print("MPS not available. Using CPU.")

model = XOR_Neural_Network(2, 8, 1).to(device)



# Training setup
EPOCHS = 1000
loss_fnc = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]).astype(np.float32)
y_xor = np.array([[0], [1], [1], [0]]).astype(np.float32)

X = torch.from_numpy(X_xor)
y = torch.from_numpy(y_xor)

X = X.to(device)
y = y.to(device)

# Training loop
for epoch in range(EPOCHS):
    y_hat = model(X)
    loss = loss_fnc(y_hat, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

for name, param in model.named_parameters():                                                                                                                   
      print(name, param.shape, param.data)

print("=" * 50 + "\n")
print("XOR Neural Network: Pytorch")
results = model(X)
print(f"XOR_Neural_Network Results: {results}")


MPS device found!
input_layer.weight torch.Size([8, 2]) tensor([[ 1.4868,  1.3814],
        [-0.2413, -0.1328],
        [ 2.7553,  2.7551],
        [-0.1340, -0.5666],
        [ 1.0642,  0.5750],
        [ 0.9680,  0.6494],
        [ 1.2133,  1.1961],
        [ 0.1452, -0.6110]], device='mps:0')
input_layer.bias torch.Size([8]) tensor([-3.3802e-04, -2.7678e-01, -2.7569e+00, -3.3859e-01,  1.3089e-01,
         1.1175e-01,  1.1969e-03,  1.2331e+00], device='mps:0')
hidden_layer.weight torch.Size([1, 8]) tensor([[ 1.8481, -0.0088, -4.6772,  0.3154,  0.7212,  0.6714,  1.5630, -1.0957]],
       device='mps:0')
hidden_layer.bias torch.Size([1]) tensor([-1.4422], device='mps:0')

XOR Neural Network: Pytorch
XOR_Neural_Network Results: tensor([[0.0677],
        [0.9651],
        [0.9637],
        [0.0252]], device='mps:0', grad_fn=<SigmoidBackward0>)


## Exercise 4: MNIST with PyTorch

Train a feedforward network on MNIST. Aim for >95% accuracy.

In [42]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Load MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST('data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:00<00:00, 26.0MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 1.76MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 14.0MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 5.01MB/s]


In [59]:
# Build a feedforward network:
#   Input: 784 (28x28 flattened)
#   Hidden 1: 128 units + ReLU
#   Hidden 2: 128 units + ReLU
#   Output: 10 units (one per digit)
#
# Use nn.CrossEntropyLoss() (combines softmax + CE)
# Use optim.Adam(model.parameters(), lr=0.001)
#
# Train for 5 epochs. Print train loss and test accuracy each epoch.
# Remember to flatten the input: x = x.view(-1, 784)

# Your code here:

class MNIST_Classifier(nn.Module):

    def __init__(self, input_dim=784, hidden1_dim=128, hidden2_dim=128, output_dim=10):
        super().__init__()
        
        self.input_layer = nn.Linear(input_dim, hidden1_dim)
        self.hidden1_layer = nn.Linear(hidden1_dim, hidden2_dim)
        self.hidden2_layer = nn.Linear(hidden2_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.input_layer(x)
        x = self.hidden1_layer(x)
        x = self.relu(x)
        x = self.hidden2_layer(x)
        x = self.relu(x)
        return x


# Model setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found!")
else:
    device = torch.device("cpu")
    print("MPS not available. Using CPU.")

model = MNIST_Classifier().to(device)



# Training setup
EPOCHS = 5
loss_fnc = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# prepare the data
train_data = datasets.MNIST('data', train=True, transform=transform)
#print(train_data[0])

# Training loop
for epoch in range(EPOCHS):
    for images, labels in train_loader:
        # move the tensors to the gpu (mps)
        images = images.to(device)
        labels = labels.to(device)

        # perform inference
        y_hat = model(images)

        # compute loss
        loss = loss_fnc(y_hat, labels)

        # back propagation
        optimizer.zero_grad()  # zeroes out the gradients -> pytorch accumulates the gradients so we have to zero them
        loss.backward()        # perform back propagation -> calculates the gradients
        optimizer.step()       # updates the parameters


# put the model in inference mode
model.eval()
correct = 0
total = 0


with torch.no_grad(): # disables gradient computation (speeds things up)
    for images, labels in test_loader:

        # perform the inference
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)  # pick the class with highest logit
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        


print("=" * 50 + "\n")
print("MNIST_Classifier:")
print(f"Accuracy: {correct/total}")






MPS device found!

MNIST_Classifier:
Accuracy: 0.9743


## Exercise 5: Overfit One Batch

Verify the model has enough capacity by memorizing a single batch.

In [63]:
# Take one batch from train_loader (use next(iter(train_loader)))
# Train for 200 iterations on ONLY that batch
# The loss should go to ~0 and accuracy on that batch should hit 100%
# If it doesn't, the model is too small or something is broken
#
# This is the first sanity check you should always do before full training

# Your code here:
images, labels = next(iter(train_loader))
EPOCHS = 200

for epoch in range(EPOCHS):
    # move the tensors to the gpu (mps)
    images = images.to(device)
    labels = labels.to(device)
    y_hat = model(images)
    loss = loss_fnc(y_hat, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


# put the model in inference mode
model.eval()
correct = 0
total = 0


with torch.no_grad(): # disables gradient computation (speeds things up)
    # perform the inference
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    predictions = outputs.argmax(dim=1)  # pick the class with highest logit
    correct += (predictions == labels).sum().item()
    total += labels.size(0)
        


print("=" * 50 + "\n")
print("MNIST_Classifier:")
print(f"Accuracy: {correct/total}")


MNIST_Classifier:
Accuracy: 1.0
